# W6 Homework: Yellow Trip Data (Nov 2025)


In [2]:
# Download the November 2025 Yellow trip data
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -O yellow_tripdata_2025-11.parquet

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("W6-Homework") \
    .getOrCreate()

# Read November 2025 Yellow trip data into a Spark DataFrame
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

# Repartition to 4 partitions and save as Parquet
df.repartition(4).write.mode("overwrite").parquet("pq/")

print("Done. Row count:", df.count())
df.printSchema()

Done. Row count: 4181444
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [6]:
df.createOrReplaceTempView("yellow_taxi")

spark.sql("SELECT * FROM yellow_taxi LIMIT 10").show()

spark.sql("SELECT COUNT(*) FROM yellow_taxi").show()



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [34]:
# Question 3

spark.sql("""
    SELECT COUNT(*) FROM yellow_taxi 
    where tpep_pickup_datetime >= '2025-11-15' and tpep_pickup_datetime < '2025-11-16'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [ ]:
# Question 4

spark.sql("""
select max(trip_duration)  FROM 
    (SELECT ( unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime) ) / 3600.0 AS trip_duration 
    FROM yellow_taxi)
"""
).show()

+------------------+
|max(trip_duration)|
+------------------+
|         90.646667|
+------------------+



In [11]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-05 00:39:55--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.167.84.228, 3.167.84.127, 3.167.84.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.167.84.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-05 00:39:55 (795 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [ ]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)


In [13]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [14]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [18]:
df_final = df.join(df_zones, df.PULocationID == df_zones.LocationID, "left")

In [19]:
df_final.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = 

In [20]:
df_final.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+

In [21]:
df_final.createOrReplaceTempView("trip_data")

spark.sql("SELECT * FROM trip_data LIMIT 10").show()

spark.sql("SELECT COUNT(*) FROM trip_data").show()



+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+

+--------+
|count(1)|
+--------+
| 4181444|
+--------+



In [ ]:
# Question 6

spark.sql("SELECT Zone, COUNT(1) from trip_data GROUP BY Zone ORDER BY COUNT(1)").show()

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|Governor's Island...|       1|
|Eltingville/Annad...|       1|
|       Arden Heights|       1|
|       Port Richmond|       3|
|       Rikers Island|       4|
|   Rossville/Woodrow|       4|
|         Great Kills|       4|
| Green-Wood Cemetery|       4|
|         Jamaica Bay|       5|
|         Westerleigh|      12|
|New Dorp/Midland ...|      14|
|             Oakwood|      14|
|        Crotona Park|      14|
|       West Brighton|      14|
|       Willets Point|      15|
|Breezy Point/Fort...|      16|
|Saint George/New ...|      17|
|       Broad Channel|      18|
|     Mariners Harbor|      21|
|Heartland Village...|      22|
+--------------------+--------+
only showing top 20 rows


In [26]:
spark.sql("SELECT PULocationID, COUNT(1) from yellow_taxi GROUP BY PULocationID ORDER BY COUNT(1)").show()





+------------+--------+
|PULocationID|count(1)|
+------------+--------+
|         105|       1|
|           5|       1|
|          84|       1|
|         187|       3|
|         204|       4|
|         199|       4|
|         111|       4|
|         109|       4|
|           2|       5|
|         251|      12|
|          59|      14|
|         176|      14|
|         172|      14|
|         245|      14|
|         253|      15|
|          27|      16|
|         206|      17|
|          30|      18|
|         156|      21|
|         118|      22|
+------------+--------+
only showing top 20 rows


In [30]:
df_zones.createOrReplaceTempView("zones")

In [32]:
spark.sql("select * from zones where LocationID in (5,84,105)").show(truncate=False)

+----------+-------------+---------------------------------------------+------------+
|LocationID|Borough      |Zone                                         |service_zone|
+----------+-------------+---------------------------------------------+------------+
|5         |Staten Island|Arden Heights                                |Boro Zone   |
|84        |Staten Island|Eltingville/Annadale/Prince's Bay            |Boro Zone   |
|105       |Manhattan    |Governor's Island/Ellis Island/Liberty Island|Yellow Zone |
+----------+-------------+---------------------------------------------+------------+

